# Contextualized Two-Tower on MovieLens 1M — Three Context Modes

End-to-end demo of `ContextualizedTwoTowerEstimator` on MovieLens 1M, illustrating all
three `context_mode` values and their serving trade-offs.

Unlike the SASRec notebooks, this is a **collaborative filtering** model — it learns
user and item embeddings jointly from ratings, without modelling item sequences.
Context features derived from rating timestamps (time of day, weekday/weekend) let the
model capture session-level preferences on top of a user's long-term taste.

## The Three Modes

Both towers produce a `final_embedding_dim`-dimensional vector. How interaction
context features are incorporated is controlled by `context_mode`:

| `context_mode` | Context placement | Score formula | ANN? | User embs offline? |
|---|---|---|---|---|
| `"user_tower"` | Concatenated into user tower input | `dot(user_tower(u, ctx), item_tower(i))` | ✅ | ❌ |
| `"trilinear"` | Hadamard product with user rep | `dot(user_rep ⊙ ctx_emb, item_rep)` | ✅ | ✅ |
| `"scoring_layer"` | Concatenated for final linear output | `linear([user_rep ‖ item_rep ‖ ctx_rep])` | ❌ | ✅ |

**Data**: downloaded automatically to `data/ml-1m/` (excluded from git).

## 1. Imports

In [1]:
import logging
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.dataset.users_dataset import UsersDataset
from skrec.estimator.embedding.contextualized_two_tower_estimator import (
    ContextualizedTwoTowerEstimator,
)
from skrec.recommender.ranking import RankingRecommender
from skrec.scorer.universal import UniversalScorer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

RAW_DIR = Path("data/movielens-1m")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/two-tower")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


## 2. Download MovieLens 1M

In [2]:
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_path = RAW_DIR / "ml-1m.zip"

if not (RAW_DIR / "ratings.dat").exists():
    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(ML1M_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith(".dat"):
                filename = Path(name).name
                with zf.open(name) as src, open(RAW_DIR / filename, "wb") as dst:
                    dst.write(src.read())
    print("Downloaded and extracted.")
else:
    print("Already downloaded.")

Already downloaded.


## 3. Load Raw Data

In [3]:
# ratings.dat: UserID::MovieID::Rating::Timestamp
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)

# users.dat: UserID::Gender::Age::Occupation::Zip-code
users_raw = pd.read_csv(
    RAW_DIR / "users.dat",
    sep="::",
    engine="python",
    names=["UserID", "Gender", "Age", "Occupation", "Zip"],
)

# movies.dat: MovieID::Title::Genres
movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1",
)

print(f"Ratings : {len(ratings):,}  |  Users : {ratings.UserID.nunique():,}  |  Movies : {ratings.MovieID.nunique():,}")
ratings.head()

Ratings : 1,000,209  |  Users : 6,040  |  Movies : 3,706


,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## 4. Feature Engineering

### User features (from `users.dat`)

- `gender_male`: 1 = M, 0 = F
- `age_norm`: age category (1/18/25/35/45/50/56) normalized to [0, 1]
- `occ_norm`: occupation code (0–20) normalized to [0, 1]

### Item features (from `movies.dat`)

Top 5 genre binary flags: **Action**, **Comedy**, **Drama**, **Romance**, **Thriller**.

### Context features (derived from rating `Timestamp`)

- `hour_norm`: hour of day (0–23) / 23 — captures time-of-day session intent
- `is_weekend`: 1 if Saturday or Sunday, 0 otherwise

These are the features the model uses as **interaction context** — they vary per
request at serving time, unlike user/item profiles which are static.

In [4]:
# ---- User features ----
users_df = pd.DataFrame(
    {
        "USER_ID": users_raw["UserID"].astype(str),
        "gender_male": (users_raw["Gender"] == "M").astype(float),
        "age_norm": users_raw["Age"] / 56.0,
        "occ_norm": users_raw["Occupation"] / 20.0,
    }
)

# ---- Item features (top-5 genre binary flags) ----
TOP_GENRES = ["Action", "Comedy", "Drama", "Romance", "Thriller"]

genre_flags = {g: movies["Genres"].str.contains(g, regex=False).astype(float) for g in TOP_GENRES}

items_df = pd.DataFrame({"ITEM_ID": movies["MovieID"].astype(str), **genre_flags})

# ---- Context features from timestamps ----
ts = pd.to_datetime(ratings["Timestamp"], unit="s")
hour_norm = ts.dt.hour / 23.0
is_weekend = (ts.dt.dayofweek >= 5).astype(float)

interactions_df = pd.DataFrame(
    {
        "USER_ID": ratings["UserID"].astype(str),
        "ITEM_ID": ratings["MovieID"].astype(str),
        "OUTCOME": (ratings["Rating"] >= 4).astype(float),  # binary: liked vs disliked
        "TIMESTAMP": ratings["Timestamp"],  # kept for split; dropped from model
        "hour_norm": hour_norm.values.round(4),
        "is_weekend": is_weekend.values,
    }
)

print(f"Users  : {len(users_df):,}  with features: {users_df.columns.tolist()}")
print(f"Items  : {len(items_df):,}  with features: {items_df.columns.tolist()}")
print(f"Interactions: {len(interactions_df):,}  positive rate: {interactions_df['OUTCOME'].mean():.1%}")
interactions_df.head()

Users  : 6,040  with features: ['USER_ID', 'gender_male', 'age_norm', 'occ_norm']
Items  : 3,883  with features: ['ITEM_ID', 'Action', 'Comedy', 'Drama', 'Romance', 'Thriller']
Interactions: 1,000,209  positive rate: 57.5%


,USER_ID,ITEM_ID,OUTCOME,TIMESTAMP,hour_norm,is_weekend
0,1,1193,1.0,978300760,0.9565,1.0
1,1,661,0.0,978302109,0.9565,1.0
2,1,914,0.0,978301968,0.9565,1.0
3,1,3408,1.0,978300275,0.9565,1.0
4,1,2355,1.0,978824291,1.0000,1.0


## 5. Train / Validation / Test Split (Leave-Last-Out)

For each user:
- **Test** = last interaction (by timestamp)
- **Validation** = second-to-last (used for early-stopping monitoring)
- **Training** = everything else

Users with fewer than 5 interactions are excluded.

**Note**: unlike the SASRec notebooks, the test item is **not** included in
training — this is a standard CF hold-out protocol, not a sequential next-item
prediction setup.

In [5]:
interactions_df = interactions_df.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

# Filter users with >= 5 interactions
counts = interactions_df.groupby("USER_ID").size()
keep_users = counts[counts >= 5].index
interactions_df = interactions_df[interactions_df["USER_ID"].isin(keep_users)].reset_index(drop=True)

# Rank from end: 0=last (test), 1=second-to-last (valid), 2+=training
interactions_df["_rank"] = interactions_df.groupby("USER_ID").cumcount(ascending=False)

test_df = interactions_df[interactions_df["_rank"] == 0].drop(columns=["_rank"]).reset_index(drop=True)
valid_df = interactions_df[interactions_df["_rank"] == 1].drop(columns=["_rank"]).reset_index(drop=True)
train_df = interactions_df[interactions_df["_rank"] >= 2].drop(columns=["_rank"]).reset_index(drop=True)

print(f"Train : {len(train_df):,}  |  Valid : {len(valid_df):,}  |  Test : {len(test_df):,}")
print(f"Users : {train_df.USER_ID.nunique():,}")

Train : 988,129  |  Valid : 6,040  |  Test : 6,040
Users : 6,040


## 6. Save CSVs and Create Datasets

In [6]:
# Drop TIMESTAMP — not a model feature; only used for splitting
train_df_model = train_df.drop(columns=["TIMESTAMP"])
valid_df_model = valid_df.drop(columns=["TIMESTAMP"])

train_path = str(DATA_DIR / "train_interactions.csv")
valid_path = str(DATA_DIR / "valid_interactions.csv")
users_path = str(DATA_DIR / "users.csv")
items_path = str(DATA_DIR / "items.csv")

if not Path(train_path).exists():
    train_df_model.to_csv(train_path, index=False)
if not Path(valid_path).exists():
    valid_df_model.to_csv(valid_path, index=False)
if not Path(users_path).exists():
    users_df.to_csv(users_path, index=False)
if not Path(items_path).exists():
    items_df.to_csv(items_path, index=False)

users_ds = UsersDataset(data_location=users_path)
items_ds = ItemsDataset(data_location=items_path)
interactions_ds = InteractionsDataset(data_location=train_path)
valid_inter_ds = InteractionsDataset(data_location=valid_path)

print(f"Training interactions : {len(train_df_model):,}")
print("Context features      : hour_norm, is_weekend")
print("Datasets created.")

Training interactions : 988,129
Context features      : hour_norm, is_weekend
Datasets created.


## 7. Train All Three Modes

We train the same architecture with each `context_mode` to compare:
- `"user_tower"` — context concatenated into the user tower (industry-standard default)
- `"trilinear"` — context modulates user representation via Hadamard product
- `"scoring_layer"` — most expressive; final linear layer over all three reps

All models use `early_stopping_patience=3` on the validation split.

In [7]:
COMMON_PARAMS = dict(
    user_embedding_dim=64,
    item_embedding_dim=64,
    final_embedding_dim=32,
    user_tower_hidden_dim1=128,
    item_tower_hidden_dim1=128,
    epochs=30,
    batch_size=512,
    learning_rate=1e-3,
    early_stopping_patience=3,
    restore_best_weights=True,
    random_state=42,
    verbose=1,
)

recommenders = {}
estimators = {}

for mode in ("user_tower", "trilinear", "scoring_layer"):
    print(f"\n{'=' * 60}")
    print(f"Training context_mode='{mode}'")
    print("=" * 60)

    est = ContextualizedTwoTowerEstimator(context_mode=mode, **COMMON_PARAMS)
    rec = RankingRecommender(UniversalScorer(est))
    rec.train(
        users_ds=users_ds,
        items_ds=items_ds,
        interactions_ds=interactions_ds,
        valid_interactions_ds=valid_inter_ds,
    )
    estimators[mode] = est
    recommenders[mode] = rec
    print(f"Done — context_mode='{mode}'")

print("\nAll three modes trained.")


Training context_mode='user_tower'


2026-04-16 22:49:48,198 - skrec.estimator.embedding.contextualized_two_tower_estimator - WARNING context_mode='user_tower' with 2 context feature(s): user embeddings are context-dependent and cannot be precomputed. get_user_embeddings() will raise. Use predict_proba_with_embeddings() at serving time, or switch to context_mode='trilinear' or 'scoring_layer' for precomputable user embeddings.


2026-04-16 22:49:48,198 skrec.estimator.embedding.contextualized_two_tower_estimator WARNING context_mode='user_tower' with 2 context feature(s): user embeddings are context-dependent and cannot be precomputed. get_user_embeddings() will raise. Use predict_proba_with_embeddings() at serving time, or switch to context_mode='trilinear' or 'scoring_layer' for precomputable user embeddings.


2026-04-16 22:49:52,842 - skrec.estimator.embedding.base_pytorch_estimator - INFO Epoch [1/30], Loss: 0.5785, Val Loss: 0.5596


2026-04-16 22:49:52,842 skrec.estimator.embedding.base_pytorch_estimator INFO Epoch [1/30], Loss: 0.5785, Val Loss: 0.5596


...

2026-04-16 22:51:43,129 - skrec.estimator.embedding.base_pytorch_estimator - INFO Epoch [11/30], Loss: 0.5355, Val Loss: 0.5432


2026-04-16 22:51:43,129 skrec.estimator.embedding.base_pytorch_estimator INFO Epoch [11/30], Loss: 0.5355, Val Loss: 0.5432


2026-04-16 22:51:43,130 - skrec.estimator.embedding.base_pytorch_estimator - INFO Early stopping at epoch 11/30 — val loss did not improve for 3 epoch(s). Best val loss: 0.5419


2026-04-16 22:51:43,130 skrec.estimator.embedding.base_pytorch_estimator INFO Early stopping at epoch 11/30 — val loss did not improve for 3 epoch(s). Best val loss: 0.5419


Done — context_mode='scoring_layer'

All three modes trained.


## 8. Evaluate: HR@10 and NDCG@10 (Sampled Ranking)

For each user, the held-out test item is ranked against **100 randomly sampled
negative items**. HR@10 and NDCG@10 are computed within these 101 candidates.

The context passed at scoring time matches the test interaction's context
(`hour_norm`, `is_weekend`) — exactly as it would arrive at a serving endpoint.

In [8]:
rng = np.random.default_rng(42)

# scorer.item_names are the same for all modes (same items_ds)
ref_scorer = recommenders["user_tower"].scorer
all_item_ids = np.array(list(ref_scorer.item_names))
known_items = set(ref_scorer.item_names)

# Test interactions: one per user, with actual request-time context
eval_test_df = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
eval_users = eval_test_df["USER_ID"].unique()

# One context row per user — USER_ID + context features only (no ITEM_ID/OUTCOME needed)
test_context_df = eval_test_df[["USER_ID", "hour_norm", "is_weekend"]].drop_duplicates("USER_ID")
user_order = test_context_df["USER_ID"].tolist()

# User interaction sets for negative sampling
all_interactions = interactions_df.copy()
user_seen = all_interactions.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()
gt_lookup = eval_test_df.set_index("USER_ID")["ITEM_ID"].to_dict()

TOP_K = 10
N_NEGATIVES = 100

results = {}
for mode, rec in recommenders.items():
    print(f"\nScoring all items for {len(user_order):,} users (mode='{mode}')...")

    # score_items: one context row per user → (n_users, n_items) score matrix
    scores_df = rec.score_items(interactions=test_context_df)
    item_cols = scores_df.columns.tolist()  # ordered list of item names
    item_to_col = {item: i for i, item in enumerate(item_cols)}
    scores_np = scores_df.to_numpy()  # (n_users, n_items)

    hits, ndcgs = [], []
    for i, user_id in enumerate(user_order):
        test_item = gt_lookup.get(user_id)
        if test_item is None or test_item not in item_to_col:
            continue

        seen = user_seen.get(user_id, set())
        neg_pool = all_item_ids[~np.isin(all_item_ids, list(seen))]
        neg_sample = rng.choice(neg_pool, size=min(N_NEGATIVES, len(neg_pool)), replace=False)

        candidate_ids = [test_item] + list(neg_sample)
        cand_cols = [item_to_col[c] for c in candidate_ids if c in item_to_col]
        cand_scores = scores_np[i, cand_cols]
        test_score = scores_np[i, item_to_col[test_item]]

        rank = int((cand_scores > test_score).sum()) + 1
        hits.append(1 if rank <= TOP_K else 0)
        ndcgs.append(1.0 / np.log2(rank + 1) if rank <= TOP_K else 0.0)

    results[mode] = {"HR@10": np.mean(hits), "NDCG@10": np.mean(ndcgs), "n": len(hits)}
    print(f"  HR@{TOP_K}: {np.mean(hits):.4f}  |  NDCG@{TOP_K}: {np.mean(ndcgs):.4f}  (n={len(hits):,})")

print(f"\n{'Mode':<16} {'HR@10':>8} {'NDCG@10':>10}")
print("-" * 36)
for mode, r in results.items():
    print(f"{mode:<16} {r['HR@10']:>8.4f} {r['NDCG@10']:>10.4f}")

2026-04-16 22:51:43,296 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


2026-04-16 22:51:43,296 skrec.scorer.universal INFO Adding Item Features for All Items, via Replication



Scoring all items for 6,040 users (mode='user_tower')...


2026-04-16 22:52:01,609 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:01,609 skrec.scorer.universal INFO Adding Item Features for All Items, via Replication


  HR@10: 0.2300  |  NDCG@10: 0.1127  (n=6,040)

Scoring all items for 6,040 users (mode='trilinear')...


2026-04-16 22:52:19,129 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:19,129 skrec.scorer.universal INFO Adding Item Features for All Items, via Replication


  HR@10: 0.2189  |  NDCG@10: 0.1055  (n=6,040)

Scoring all items for 6,040 users (mode='scoring_layer')...


  HR@10: 0.2056  |  NDCG@10: 0.0954  (n=6,040)

Mode                HR@10    NDCG@10
------------------------------------
user_tower         0.2300     0.1127
trilinear          0.2189     0.1055
scoring_layer      0.2056     0.0954


## 9. Context Sensitivity

The same user, different context: **morning** (hour_norm=0.0, is_weekend=0)
vs **weekend evening** (hour_norm=0.9, is_weekend=1).

A context-aware model should produce different top-10 lists. `user_tower` and
`trilinear` both support ANN — the query vector shifts with context. `scoring_layer`
runs full inference but has the most freedom to interact user × item × context.

In [9]:
movie_title = movies.set_index(movies["MovieID"].astype(str))["Title"].to_dict()

# Pick the first user from the test set
probe_user = user_order[0]

morning_ctx = pd.DataFrame([{"USER_ID": probe_user, "hour_norm": 0.0, "is_weekend": 0.0}])
evening_ctx = pd.DataFrame([{"USER_ID": probe_user, "hour_norm": 0.9, "is_weekend": 1.0}])

print(f"Top-10 recommendations for user {probe_user}\n")
for mode, rec in recommenders.items():
    morning_recs = rec.recommend(interactions=morning_ctx, top_k=10)[0]
    evening_recs = rec.recommend(interactions=evening_ctx, top_k=10)[0]

    # Jaccard overlap: how similar are the two lists?
    overlap = len(set(morning_recs) & set(evening_recs)) / len(set(morning_recs) | set(evening_recs))

    print(f"  [{mode}]  overlap: {overlap:.0%}")
    print(f"    Morning (weekday):  {[movie_title.get(i, i) for i in morning_recs[:5]]}")
    print(f"    Evening (weekend):  {[movie_title.get(i, i) for i in evening_recs[:5]]}")
    print()

2026-04-16 22:52:38,557 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:38,557 skrec.scorer.universal INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:38,564 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:38,564 skrec.scorer.universal INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:38,570 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


...

2026-04-16 22:52:38,581 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:38,581 skrec.scorer.universal INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:38,587 - skrec.scorer.universal - INFO Adding Item Features for All Items, via Replication


2026-04-16 22:52:38,587 skrec.scorer.universal INFO Adding Item Features for All Items, via Replication


Top-10 recommendations for user 1

  [user_tower]  overlap: 54%
    Morning (weekday):  ['Boys of St. Vincent, The (1993)', 'General, The (1927)', '12 Angry Men (1957)', 'Face in the Crowd, A (1957)', 'War Room, The (1993)']
    Evening (weekend):  ['Wrong Trousers, The (1993)', 'Face in the Crowd, A (1957)', 'Boys of St. Vincent, The (1993)', "Microcosmos (Microcosmos: Le peuple de l'herbe) (1996)", '12 Angry Men (1957)']

  [trilinear]  overlap: 11%
    Morning (weekday):  ['North by Northwest (1959)', 'For All Mankind (1989)', '12 Angry Men (1957)', 'Thin Man, The (1934)', 'Apple, The (Sib) (1998)']
    Evening (weekend):  ['Guns of Navarone, The (1961)', 'Patriot Games (1992)', 'Hustler, The (1961)', 'Rock, The (1996)', 'For All Mankind (1989)']

  [scoring_layer]  overlap: 100%
    Morning (weekday):  ['Apple, The (Sib) (1998)', 'Sanjuro (1962)', 'I Am Cuba (Soy Cuba/Ya Kuba) (1964)', 'Close Shave, A (1995)', 'Lamerica (1994)']
    Evening (weekend):  ['Apple, The (Sib) (1998)', '

## 10. Embedding Precomputation

In `trilinear` and `scoring_layer` modes the user tower is **context-free** —
user embeddings can be computed once and cached (e.g., in a feature store).

In `user_tower` mode with context features, `get_user_embeddings()` raises a
`NotImplementedError` because there is no single offline vector to precompute:
each request produces a different `user_rep` depending on the live context.

In [10]:
print("Embedding extraction per mode:\n")

for mode, est in estimators.items():
    try:
        embs = est.get_user_embeddings()
        print(f"  [{mode}]  ✓ user embeddings shape: {embs.shape}")
        print(f"           (precomputable offline — one {est.final_embedding_dim}-d vector per user)")
    except NotImplementedError as e:
        print(f"  [{mode}]  ✗ NotImplementedError (expected):")
        print(f"           {str(e)[:120]}...")
    print()

Embedding extraction per mode:

  [user_tower]  ✗ NotImplementedError (expected):
           Cannot precompute user embeddings when context_mode='user_tower' and the model was trained with context features — user ...

  [trilinear]  ✓ user embeddings shape: (6040, 2)
           (precomputable offline — one 32-d vector per user)

  [scoring_layer]  ✓ user embeddings shape: (6040, 2)
           (precomputable offline — one 32-d vector per user)



## 11. Early Stopping in Detail

The training loop monitors val loss every epoch. With `early_stopping_patience=3`,
it stops once 3 consecutive epochs pass without improvement and restores the
best weights.

Below we re-train a `trilinear` model with verbose logging turned on so you can
see the `Val Loss` lines and the `Early stopping` trigger.

In [11]:
logging.getLogger("recommender").setLevel(logging.INFO)

est_es = ContextualizedTwoTowerEstimator(
    context_mode="trilinear",  # string and ContextMode enum are both accepted
    user_embedding_dim=64,
    item_embedding_dim=64,
    final_embedding_dim=32,
    epochs=50,  # generous upper bound — early stopping fires before this
    batch_size=512,
    learning_rate=1e-3,
    early_stopping_patience=3,
    restore_best_weights=True,
    random_state=42,
    verbose=1,
)

RankingRecommender(UniversalScorer(est_es)).train(
    users_ds=users_ds,
    items_ds=items_ds,
    interactions_ds=interactions_ds,
    valid_interactions_ds=valid_inter_ds,
)
print("\nEarly-stopping demo complete.")

2026-04-16 22:52:42,728 - skrec.estimator.embedding.base_pytorch_estimator - INFO Epoch [1/50], Loss: 0.5796, Val Loss: 0.5604


2026-04-16 22:52:42,728 skrec.estimator.embedding.base_pytorch_estimator INFO Epoch [1/50], Loss: 0.5796, Val Loss: 0.5604


2026-04-16 22:52:46,216 - skrec.estimator.embedding.base_pytorch_estimator - INFO Epoch [2/50], Loss: 0.5398, Val Loss: 0.5454


2026-04-16 22:52:46,216 skrec.estimator.embedding.base_pytorch_estimator INFO Epoch [2/50], Loss: 0.5398, Val Loss: 0.5454


2026-04-16 22:52:49,658 - skrec.estimator.embedding.base_pytorch_estimator - INFO Epoch [3/50], Loss: 0.5312, Val Loss: 0.5396


...

2026-04-16 22:53:10,599 - skrec.estimator.embedding.base_pytorch_estimator - INFO Epoch [9/50], Loss: 0.5026, Val Loss: 0.5364


2026-04-16 22:53:10,599 skrec.estimator.embedding.base_pytorch_estimator INFO Epoch [9/50], Loss: 0.5026, Val Loss: 0.5364


2026-04-16 22:53:10,600 - skrec.estimator.embedding.base_pytorch_estimator - INFO Early stopping at epoch 9/50 — val loss did not improve for 3 epoch(s). Best val loss: 0.5342


2026-04-16 22:53:10,600 skrec.estimator.embedding.base_pytorch_estimator INFO Early stopping at epoch 9/50 — val loss did not improve for 3 epoch(s). Best val loss: 0.5342



Early-stopping demo complete.


## 12. Summary

### When to Use Each Mode

| Mode | Best for | ANN-compatible | User embs offline |
|---|---|---|---|
| `user_tower` | Standard two-tower serving — context drives request-time query vector | ✅ | ❌ |
| `trilinear` | Cache user embs; cheap runtime modulation before ANN search | ✅ | ✅ |
| `scoring_layer` | Maximum expressiveness; full inference at serving | ❌ | ✅ |

### Serving Patterns

```
# user_tower (ANN)
q = user_tower(user_emb, user_features, request_context)   # at request time
candidates = faiss_index.search(q, top_k)                  # prebuilt item index

# trilinear (ANN + cached user embs)
user_rep = user_tower(user_emb, user_features)             # offline, cached
ctx_emb  = context_projection(request_context)             # at request time
q        = user_rep * ctx_emb                              # Hadamard product
candidates = faiss_index.search(q, top_k)

# scoring_layer (full inference, no ANN)
user_rep = user_tower(user_emb, user_features)             # offline, cached
ctx_emb  = context_projection(request_context)             # at request time
for each candidate item:
    score = linear_layer([user_rep, item_rep, ctx_emb])
```

### API Notes

- `context_mode` accepts the `ContextMode` enum **or** the equivalent string (`"trilinear"`).
- `get_config()` always serializes `context_mode` as a plain string so configs
  round-trip through JSON / pickle without importing `ContextMode`.
- At scoring time, pass one context row per user to `recommender.score_items()` or
  `recommender.recommend()`. The scorer internally cross-joins with the item catalog
  to produce the `(n_users × n_items)` score matrix.
- `TIMESTAMP` (used only for splitting) should be **dropped before creating datasets**
  so it is not treated as a context feature.